<img src="https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/ressources/img/logo_macmia.png" alt="Banque des Territoires · France 2030 · MACMIA" width="520">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees_v2/exercices/seance1_exercices.ipynb)

# Séance 2.1 — Charger, comprendre et nettoyer une base de données

**Exercices** · durée : 6h (2h de cours, 4h d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- charger un fichier de données depuis le web en une ligne
- décrire un fichier que vous n'avez jamais vu en 30 secondes
- sélectionner exactement les lignes et les colonnes qui vous intéressent
- calculer des indicateurs simples sur une colonne entière
- lire un message d'erreur au lieu de le subir

## Exercice — Où acheter à Paris ?

## La question

Vous êtes analyste dans une agence immobilière parisienne. Un client dispose de
**400 000 €** et vous pose une question simple :

> *« Dans quel arrondissement est-ce que j'achète le plus de mètres carrés ? »*

Pour répondre, l'agence vous remet l'export brut de **toutes les ventes
immobilières enregistrées à Paris en 2024**. Ce sont des données publiques,
publiées par l'administration fiscale, et elles sont dans l'état où on les
reçoit : personne ne les a nettoyées avant vous.

À la fin de cet exercice, vous aurez :

- un fichier propre, dont vous saurez dire ligne par ligne ce que vous avez retiré et pourquoi ;
- le prix au mètre carré de chacun des vingt arrondissements ;
- la réponse au client, chiffrée ;
- et une carte de Paris colorée par le prix, que vous aurez produite vous-même.

## Comment ça marche

L'exercice se fait seul, en quatre parties, dans l'ordre. Comptez une demi-heure
pour la première, une heure et demie pour la deuxième, une heure pour la
troisième, une demi-heure pour la dernière.

**Chaque exercice est une cellule vide que vous écrivez entièrement.** Juste
avant, un encadré *Rappel* vous redonne la forme de la commande vue en cours.
Vous n'avez rien à deviner : vous avez à l'appliquer à ce fichier.

Vous rencontrerez aussi trois autres sortes de cellules :

- des **cellules à exécuter telles quelles** : le code est déjà écrit, il vous montre quelque chose ;
- des **cellules de vérification**, aux moments où une erreur fausserait toute la suite. Elles affichent `OK` ou `A REVOIR` avec un indice ;
- des **cellules de prédiction** : on vous demande d'écrire ce que vous attendez, en commentaire, *avant* d'exécuter la cellule suivante. Prenez-les au sérieux, c'est là qu'on apprend.

Si une vérification affiche `NameError`, c'est que la cellule d'exercice
au-dessus n'a pas été exécutée, ou qu'elle contient une faute. Corrigez-la,
relancez-la, puis relancez la vérification.

Tout ce dont vous avez besoin est dans la séance pandas et dans la séance 1.1.
Quand un outil nouveau est nécessaire, il est présenté sur place.

## Partie 0 — Mise en route

La cellule de setup est la même que dans tous les notebooks du cours. Exécutez-la
en premier.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://cdn.jsdelivr.net/gh/maxischa/datacamp_test@bf9431e/bloc2_donnees_v2/data/"

In [ ]:
def verifier(nom, condition, indice=""):
    """Affiche un retour immediat sans interrompre le notebook."""
    print("OK   -", nom) if condition else print("A REVOIR -", nom, ":", indice)

In [ ]:
brut = pd.read_csv(BASE + "immo_paris_sale.csv")   ## le fichier brut, tel que recu

---

## Partie 1 — Découvrir le fichier

Avant de nettoyer quoi que ce soit, il faut savoir ce qu'on a entre les mains.
Le réflexe du cours, toujours dans cet ordre : `shape`, `info()`, `head()`,
`describe()`. Trente secondes, et vous savez de quoi vous parlez.

### Exercice 1 — La carte d'identité du fichier

Affichez la taille du tableau, ses colonnes avec leur type, et ses cinq
premières lignes. Puis, en commentaire à la fin de la cellule, répondez à la
question : **quelles colonnes devraient contenir des nombres, et n'en
contiennent pas ?** Regardez la colonne `Dtype` de `info()` : `object` veut
dire texte.

> **Rappel.** Dans une même cellule, seule la dernière expression s'affiche
> toute seule : mettez les autres dans un `print()`, ou une par cellule.

Dix colonnes. Vous les retrouverez tout au long de l'exercice :

| Colonne | Contenu |
|---|---|
| `vente_id` | l'identifiant de la vente |
| `date` | la date de la vente |
| `prix` | le prix payé, en euros |
| `rue` | le nom de la rue |
| `arrondissement` | de 1 à 20 |
| `type_local` | ce qui est vendu : appartement, dépendance, local commercial, maison |
| `surface` | la surface bâtie, en m² |
| `pieces` | le nombre de pièces principales |
| `longitude`, `latitude` | la position sur la carte |

### Exercice 2 — La photo chiffrée

`describe()` donne pour chaque colonne de nombres ses huit statistiques :
effectif, moyenne, écart-type, minimum, les trois quartiles et maximum.
Affichez-le, arrondi à une décimale, puis répondez aux deux questions en
commentaire :

1. Une colonne que vous attendiez n'apparaît pas dans le tableau. Laquelle, et
   pourquoi ?
2. Regardez le `max` de `surface`. Est-ce une surface d'appartement ?

Deux choses à retenir de ce tableau.

`prix` n'y est pas, et **pandas ne s'en est pas plaint**. `describe()` ne
traite que les colonnes de nombres ; une colonne de texte est simplement
laissée de côté, sans message.

`surface` a un maximum de plusieurs dizaines de milliers de m². Personne
n'habite cinq hectares : le fichier ne contient donc pas que des appartements.

### Exercice 3 — Une ligne, c'est quoi ?

C'est la question à se poser devant n'importe quel fichier. Ici, une
ligne est-elle une vente ? Pour le savoir, comparez le nombre de lignes au
nombre de valeurs **distinctes** de `vente_id`.

> **Rappel.** `len` ou `shape` comptent les lignes.
> `nunique` compte les valeurs distinctes.

In [ ]:
verifier("nombre de lignes", nb_lignes == 61276, "len(brut)")
verifier("nombre de ventes distinctes", nb_ventes == 35631, "nunique() sur la colonne vente_id")

Presque deux fois plus de lignes que de ventes. Une ligne n'est donc pas une
vente. Regardons une vente précise pour comprendre. Exécutez la cellule telle
quelle :

In [ ]:
brut.query("vente_id == '2024-1193275'")   ## une seule vente, toutes ses lignes

Une vente, trois lignes : un appartement, et deux dépendances (une cave, un
parking, ce genre de lots). **Chaque ligne est un lot**, et une vente peut en comporter
plusieurs. Le `prix` est le même sur les trois lignes : c'est le prix de la
vente entière, pas celui du lot.

Retenez-le, c'est le piège principal de ce fichier. Pour calculer un prix au
mètre carré d'appartement, il faudra isoler les ventes qui ne contiennent
qu'un seul appartement. Ce sera la dernière étape du nettoyage.

Remarquez aussi comment `type_local` est écrit d'une ligne à l'autre.

### Exercice 4 — Compter les types de biens

Comptez les valeurs de `type_local` avec `value_counts()`, après mettez le nombre
d'écritures différentes dans `nb_ecritures`. Combien de types de biens existent
**vraiment** ?

> **Rappel.** `value_counts` pour avoir les effectifs par catégorie, sous la forme
> d'un tableau trié. Le nombre de lignes de ce tableau donne le nombre d'écritures.

In [ ]:
verifier("nombre d'ecritures", nb_ecritures == 16, "len() sur le resultat de value_counts()")

Seize écritures pour quatre types réels : `Appartement`, `Dépendance`, `Local
industriel. commercial ou assimilé`, `Maison`. Les autres sont les mêmes mots
avec un espace devant, un espace derrière, ou en majuscules. Pour pandas, ce
sont seize catégories différentes. On réparera ça en partie 2.

### Exercice 5 — Le prix moyen

Calculez le prix moyen des ventes du fichier.

**Ça va échouer.** Exécutez quand même, puis lisez la dernière ligne du message
et écrivez en commentaire, dans la cellule d'après, pourquoi ça ne marche pas.
Vous avez tous les éléments depuis l'exercice 1.

`TypeError`, et le message parle de texte (`string`) qu'il ne peut pas
convertir en nombre.

Vous ne pouvez rien calculer sur ce fichier tant qu'il n'est pas nettoyé. C'est
le programme de la partie 2.

---

## Partie 2 — Nettoyer

Sept défauts, un par section, toujours traités de la même façon :

1. **constater** : compter le défaut ;
2. **comprendre** : d'où vient-il, et qu'est-ce que ça change ;
3. **corriger** ;
4. **mesurer** : combien de lignes ont disparu, noté dans un journal ;
5. **vérifier**.

Le fichier nettoyé s'appelle `propre`. Le fichier brut reste `brut`, on ne le
modifie jamais : si quelque chose se passe mal, on repart de lui.

Le journal est une liste, comme à la séance 1.1. On y ajoute une phrase à chaque
étape, et on l'imprimera à la fin.

In [ ]:
journal = []   ## une phrase par etape de nettoyage

### Défaut 1 — Les doublons

On commence toujours par là. Tant qu'une ligne peut être la copie d'une autre,
aucun compte ne veut rien dire : on croirait retirer 100 ventes sans client
alors qu'on en retire 80 et 20 copies.

### Exercice 6 — Dédoublonner

Comptez les lignes strictement identiques à une ligne précédente, mettez le
résultat dans `nb_doublons`, puis créez `propre` : le fichier brut sans ces
copies. Ajoutez une phrase au journal.

> **Rappel.** `df.duplicated()` rend un Vrai/Faux par ligne, `.sum()` compte
> les Vrai. `df.drop_duplicates()` retire les copies. Ajoutez `.copy()` après
> un filtrage, pour avoir un vrai tableau à soi.
>
> **Rappel.** `journal.append(f"doublons : {nb_doublons} lignes")` ajoute une
> phrase à la liste.

In [ ]:
verifier("doublons retires", len(propre) == 59492, "drop_duplicates() sur brut, puis .copy()")

### Défaut 2 — Les valeurs manquantes

### Exercice 7 — Compter les trous

Comptez les valeurs manquantes de chaque colonne de `propre`, et mettez le
résultat dans `trous`.

> **Rappel.** `df.isna().sum()` : un compte de trous par colonne.

In [ ]:
verifier("surfaces manquantes", trous["surface"] == 24297, "isna().sum() sur propre, puis la ligne surface")

Plus de 24 000 surfaces manquantes, soit **40 % du fichier**. Le réflexe
serait de les supprimer. Ce serait une erreur de méthode.

La question du cours : **pourquoi cette valeur manque-t-elle ?** On ne
supprime pas un trou sans savoir ce qu'il y a derrière. La cellule suivante
marque les lignes sans surface dans une colonne, puis n'affiche qu'elles.
Exécutez-la telle quelle et regardez la colonne `type_local`.

In [ ]:
propre["sans_surface"] = propre["surface"].isna()      ## Vrai si la surface manque
propre.query("sans_surface == True").head(8)           ## on ne garde que ces lignes

Des dépendances, sous leurs différentes écritures. Une cave ou un parking n'a
pas de surface habitable : **le trou est normal**, il ne signale aucune erreur.

Et surtout, ces lignes ne nous intéressent pas. Notre question porte sur des
appartements. Si on supprimait les surfaces manquantes maintenant, on
retirerait les bonnes lignes pour la mauvaise raison, et le journal dirait
« 24 000 lignes perdues pour cause de données manquantes », ce qui est faux.

**On ne supprime rien ici.** On traitera les types de biens au défaut 5, et on
reviendra alors sur les trous qui restent. La colonne de marquage ne sert
plus, on la retire :

In [ ]:
propre = propre.drop(columns=["sans_surface"])   ## drop(columns=[...]) retire une colonne

### Défaut 3 — Les nombres stockés en texte

Regardez à quoi ressemble la colonne `prix` :

In [ ]:
propre["prix"].head(8)

Deux problèmes, comme en cours, mais pas tout à fait les mêmes : une
**virgule** décimale là où Python attend un point, et un suffixe sur certaines
valeurs. Regardez bien lequel : ce n'est pas celui du fichier de cours.

### Exercice 8 — Le prix en nombre

Avant d'écrire, une prédiction. À l'exercice 7, `trous` vous a dit combien de
prix manquaient déjà dans le fichier. Une conversion réussie ne doit en perdre
**aucun de plus**. Combien de `NaN` attendez-vous après conversion ?

In [ ]:
# Ma prediction (nombre de NaN attendus apres conversion) :

Maintenant convertissez : retirez le suffixe, remplacez la virgule par un
point, convertissez en nombre, remettez le résultat dans `propre["prix"]`. Puis
comptez les `NaN` obtenus dans `nb_prix_perdus`.

> **Rappel.** Les trois étapes du cours :
> ```python
> txt = df["prix"].str.replace(" EUR", "")     # 1. le suffixe (adaptez-le !)
> txt = txt.str.replace(",", ".")              # 2. la virgule
> df["prix"] = pd.to_numeric(txt, errors="coerce")   # 3. la conversion
> ```
> `errors="coerce"` met `NaN` quand il ne sait pas convertir, sans prévenir.
> D'où le compte obligatoire juste après.

In [ ]:
verifier("prix en nombres", propre["prix"].dtype == "float64", "pd.to_numeric sur le texte nettoye")
verifier("aucun prix perdu", nb_prix_perdus == 79,
         "si vous en avez pres de 8937, le suffixe n'a pas ete retire : regardez-le dans les donnees, ce n'est pas ' EUR'")

Si votre vérification a échoué avec près de 9 000 prix perdus, vous venez de
vivre le danger de `errors="coerce"` : **aucune erreur, aucun message, et 15 %
des prix disparus**. Corrigez le suffixe, relancez, recomptez. Ce compte après
conversion n'est pas une option.

### Défaut 4 — Les dates

Le défaut le plus piégeux du cours, et ici le piège est plus dur. On procède
comme en cours, en quatre temps.

**Temps 0 : se donner un repère avant de convertir.** La colonne est du texte
régulier : `16/02/2024`, `12.12.2024`. Quel que soit le séparateur, les
caractères 3 et 4 sont le mois. On peut donc compter les ventes par mois
**sans rien convertir**, et garder ce compte comme référence.

In [ ]:
mois_texte = propre["date"].str[3:5].value_counts().sort_index()   ## le mois, lu dans le texte
mois_texte

Douze mois, et un creux net en août. C'est notre repère : après conversion,
on doit retrouver **exactement** ces nombres.

**Temps 1 : la façon naïve.** Cellule volontairement fausse, lisez le message.

In [ ]:
pd.to_datetime(propre["date"])   ## sans format, pandas devine... et echoue

Le fichier mélange deux écritures et pandas veut un format unique. Le message
suggère lui-même `format="mixed"`.

**Temps 2 : avec `format="mixed"`.** Plus d'erreur. Regardons si le résultat
est plausible, avec la vérification habituelle du cours, puis avec notre
repère.

In [ ]:
essai = pd.to_datetime(propre["date"], format="mixed")   ## sans dayfirst

print("plus ancienne :", essai.min().date(), "| plus recente :", essai.max().date())

# Notre repere : les ventes par mois, lues dans le texte contre lues apres conversion
comparaison = pd.DataFrame({"dans le texte": mois_texte.values,
                            "apres conversion": essai.dt.month.value_counts().sort_index().values},
                           index=range(1, 13))
comparaison

Le minimum et le maximum sont parfaitement plausibles : de janvier à décembre
2024. En cours, cette vérification suffisait. **Ici, elle ne voit rien.**

Mais la comparaison avec le repère, elle, est sans appel : les nombres ne
correspondent pas, et août a gagné plusieurs centaines de ventes. Un tiers des
dates ont changé de mois. `12/03/2024` a été lu à l'américaine, mois d'abord :
le 3 décembre au lieu du 12 mars.

La leçon dépasse ce fichier : **une conversion qui ne produit pas d'erreur
n'est pas une conversion réussie**, et le minimum et le maximum ne suffisent
pas toujours à le voir. Il faut s'être donné un repère avant.

### Exercice 9 — Les dates, correctement

**Temps 3.** Convertissez `propre["date"]` en vraies dates, en imposant la
lecture française. Puis recomptez les ventes par mois à partir des dates
converties, mettez ce compte dans `mois`, et le nombre de ventes d'août dans
`nb_aout`. Il doit être égal à celui du repère.

> **Rappel.** `pd.to_datetime(col, format="mixed", dayfirst=True)`. Une fois la
> colonne convertie, `.dt.month` donne le mois de chaque ligne.
>
> **Sur une colonne de comptes.** `mois.loc[8]` va chercher la valeur
> dont l'étiquette est 8. C'est le même `.loc` que sur un tableau : on donne
> l'étiquette, il rend la valeur.

In [ ]:
verifier("dates converties", str(propre["date"].dtype).startswith("datetime"), "pd.to_datetime avec format='mixed'")
verifier("aout retrouve", nb_aout == 2257, "sans dayfirst=True, aout affiche 3102 : le jour et le mois sont inverses")

### Défaut 5 — Du texte incohérent

Retour sur `type_local` et ses seize écritures pour quatre types. Un espace
devant, un espace derrière, des majuscules : trois défauts que deux commandes
réparent.

### Exercice 10 — Uniformiser les types

Enlevez les espaces autour et passez tout en minuscules, dans
`propre["type_local"]`. Mettez le nombre de types restants dans `nb_types`.

> **Rappel.** `str.strip()` retire les espaces, `str.lower()` met la chaîne de caractères en minuscules.
> `nunique()` compte le nombre de valeurs uniques.

In [ ]:
verifier("quatre types", nb_types == 4, "strip() enleve les espaces des deux cotes, lower() la casse : il faut les deux")

### Exercice 11 — Ne garder que les appartements

Notre question porte sur des appartements. Les caves, les parkings, les
boutiques et les maisons sortent du périmètre.

Gardez dans `propre` les seules lignes dont le type est `appartement`. Mettez
le nombre de lignes obtenu dans `nb_app` : ce nombre servira au bilan. Notez
au journal les lignes écartées, en précisant bien qu'il s'agit d'un choix de
périmètre.

> **Rappel.** `df.query(condition)`. La condition, avec des guillemets doubles
> dehors, simples dedans.

In [ ]:
verifier("appartements seuls", nb_app == 30833, "query sur type_local == 'appartement', apres strip et lower")

### Exercice 12 — Retour sur les valeurs manquantes

On avait laissé les trous de côté au défaut 2, en promettant d'y revenir une
fois les dépendances écartées. Recomptez les valeurs manquantes de `propre`.
Il n'en reste qu'une poignée, sur `prix` et `surface` : ce sont de vraies
lacunes, sur des appartements. Sans prix ou sans surface, pas de prix au m² :
retirez ces lignes, et notez-les au journal.

Pourquoi ne pas les remplir, par la moyenne par exemple ? Parce qu'un prix au
m² calculé sur une surface inventée serait un chiffre inventé. Remplir un trou,
c'est affirmer quelque chose ; ici on n'a rien à affirmer.

> **Rappel.** `df.isna().sum()` : un compte de trous par colonne. `dropna(subset=["a", "b"])` retire les
> lignes où l'une de ces colonnes ("a" ou "b") est vide.

In [ ]:
verifier("lignes completes", len(propre) == 30794, "dropna(subset=['prix', 'surface'])")

### Défaut 6 — Les valeurs aberrantes

Maintenant que `prix` est un nombre, `describe()` peut enfin le voir. Exécutez :

In [ ]:
propre[["prix", "surface"]].describe().round(0)

Lisez `min` et `max`. Un appartement vendu **1 €**, un autre plusieurs
centaines de millions ; une surface de **2 m²**, une autre de plus de 1 000 m².
Et regardez la moyenne du prix face à sa médiane : la moyenne est plusieurs
fois plus haute. C'est la signature d'une poignée de valeurs énormes qui la
tirent vers le haut.

Ces valeurs ne sont pas des fautes de frappe. Un euro, c'est une vente
symbolique entre membres d'une famille ; cent millions, c'est un immeuble
entier enregistré comme un seul lot ; deux mètres carrés, c'est une chambre de
service. Ce sont de vraies transactions, mais elles ne disent rien sur le prix
d'un appartement parisien. On les écarte, et on note qu'on l'a fait.

### Exercice 13 — Le prix au mètre carré

Créez la colonne `prix_m2` : le prix divisé par la surface. C'est elle qui
répondra au client.

> **Rappel.** `df["c"] = df["a"] / df["b"]` : une seule ligne, calculée pour
> toutes les lignes.

Avant de fixer un seuil, regardez les extrêmes. `sort_values` trie un tableau
selon une colonne ; `head(3)` garde les trois premiers. Exécutez les deux
cellules :

In [ ]:
propre.sort_values("prix_m2").head(3)[["prix", "surface", "prix_m2", "arrondissement"]]   ## les moins chers

In [ ]:
propre.sort_values("prix_m2", ascending=False).head(3)[["prix", "surface", "prix_m2", "arrondissement"]]   ## les plus chers

### Exercice 14 — Borner

On garde les appartements d'au moins **9 m²** (c'est le minimum légal pour
louer un logement) et dont le prix au m² est compris entre **2 000 et 30 000 €**,
la fourchette du marché parisien. Tout le reste est écarté, et noté au journal.

> **Rappel.** Un encadrement s'écrit d'un seul tenant, et deux conditions se combinent avec `and`.

In [ ]:
verifier("valeurs bornees", len(propre) == 26778, "surface >= 9 and 2000 <= prix_m2 <= 30000, dans une seule chaine")

### Défaut 7 — Un lot n'est pas une vente

Souvenez-vous de l'exercice 3 : une vente peut contenir plusieurs lots, et le
prix inscrit est celui de la vente entière. On a écarté les caves et les
parkings, mais il reste un cas : **une vente qui contient deux appartements**,
ou plus. Deux lignes, un seul prix pour les deux. Le `prix_m2` calculé sur
chacune est faux, et on n'a aucun moyen de répartir le prix entre elles.

La seule décision honnête : ne garder que les ventes qui ne contiennent qu'un
appartement.

**Un outil nouveau.** `df.duplicated(subset=["vente_id"])` compare les lignes
sur cette seule colonne, et marque la deuxième occurrence, la troisième, etc.
Ce n'est pas ce qu'on veut : on veut écarter **toutes** les lignes d'une vente
répétée, la première comprise. C'est ce que fait l'option `keep=False` : elle
marque toutes les occurrences.

```python
df.duplicated(subset=["vente_id"], keep=False)   # Vrai sur TOUTES les lignes d'une vente repetee
```

### Exercice 15 — Isoler les ventes d'un seul appartement

Marquez les lignes dont le `vente_id` apparaît plusieurs fois, dans une colonne
`multi`. Comptez-les dans `nb_multi`. Puis ne gardez que les lignes où `multi`
est faux, et notez au journal.

Pour finir, la vérification qui referme la question de l'exercice 3 : le nombre
de lignes de `propre` doit maintenant être **égal** au nombre de ventes
distinctes.

> **Rappel.** Marquer dans une colonne, puis filtrer avec
> `query("multi == False")`, comme au défaut 2.

In [ ]:
verifier("ventes d'un seul appartement", len(propre) == 25209, "duplicated(subset=['vente_id'], keep=False), puis garder multi == False")
verifier("une ligne = une vente", len(propre) == propre["vente_id"].nunique(), "il reste des ventes repetees")

### Le bilan

Le journal, imprimé par une boucle :

In [ ]:
for ligne in journal:
    print(ligne)

print()
print("fichier recu :", len(brut), "lignes | fichier propre :", len(propre), "lignes")

### Exercice 16 — Le taux de perte

Plus de la moitié des lignes ont disparu. Le cours dit qu'au-delà de 40 % de
pertes, il faut reprendre le pipeline. Faut-il le reprendre ?

Non, et il faut savoir l'expliquer. La plus grosse ligne du journal n'est pas
une perte : c'est un **choix de périmètre**. Les caves et les boutiques ne sont
pas des données abîmées, ce sont des données qui ne concernent pas la
question. Le vrai taux de perte se mesure **sur les appartements** : parmi les
`nb_app` appartements de l'exercice 11, combien ont été écartés parce qu'ils
étaient inexploitables ?

Calculez ce taux en pourcentage, arrondi à une décimale, dans `taux_perte`.

> **Rappel.** `100 * (1 - apres / avant)`, puis `round(..., 1)`.

In [ ]:
verifier("taux de perte", taux_perte == 18.2, "comparez le fichier final a nb_app, pas au fichier brut")

Moins de 20 %, et chaque ligne écartée a sa raison dans le journal. C'est ce
que vous présenteriez à votre responsable : pas « j'ai nettoyé les données »,
mais « voici ce que j'ai retiré, et pourquoi ».

Vous avez fini la partie la plus longue. Le fichier est propre, et vous savez
exactement ce qu'il contient : **une ligne, un appartement, une vente**, avec
un prix au mètre carré fiable. Tout ce qui suit va vite.

---

> 📘 **À faire avant cette partie : la séance 2.2, en autonomie.**
>
> Tout ce qui précède se fait avec ce que la séance 2.1 vous a montré. **La
> suite, non.** Les parties 3 et 4 tracent des graphiques — un histogramme,
> des barres, une courbe, un nuage de points — et calculent une corrélation ;
> la séance 2.1 n'en montre aucun.
>
> C'est la séance **2.2 — Agréger, croiser et visualiser** qui les enseigne,
> exercices compris. Comptez deux heures, et faites-la avant d'attaquer
> l'exercice 17.
>
> Elle est dans votre **dossier Drive**, sous « 2.2 - Cours - Agreger croiser
> et visualiser » — c'est le chemin le plus court. Sinon,
> [▶ ouvrez-la sur Colab](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc2_donnees_v2/cours/seance2_cours.ipynb).

---

## Partie 3 — Analyser et tracer

### Cellule de rattrapage

Si votre nettoyage n'a pas abouti, ou si vous avez un doute, exécutez la cellule
ci-dessous : elle refait tout le nettoyage d'un coup, dans l'ordre, et produit
le même `propre`. Si tout est allé bien, vous pouvez l'exécuter quand même, elle
ne change rien.

C'est aussi, en douze lignes, le résumé de la partie 2. Relisez-la : c'est un
pipeline, qu'on peut rejouer sur le fichier de l'année prochaine.

In [ ]:
propre = pd.read_csv(BASE + "immo_paris_sale.csv").drop_duplicates().copy()
propre["prix"] = pd.to_numeric(propre["prix"].str.replace(" €", "").str.replace(",", "."), errors="coerce")
propre["date"] = pd.to_datetime(propre["date"], format="mixed", dayfirst=True)
propre["type_local"] = propre["type_local"].str.strip().str.lower()
propre = propre.query("type_local == 'appartement'").copy()
propre = propre.dropna(subset=["prix", "surface"]).copy()
propre["prix_m2"] = propre["prix"] / propre["surface"]
propre = propre.query("surface >= 9 and 2000 <= prix_m2 <= 30000").copy()
propre["multi"] = propre.duplicated(subset=["vente_id"], keep=False)
propre = propre.query("multi == False").copy()

print(propre.shape)   ## (25209, 12)

### Exercice 17 — La répartition des prix au m²

Une moyenne donne un chiffre ; un histogramme donne la forme. Tracez
l'histogramme de `prix_m2` en 50 classes, avec un titre et l'unité en abscisse.

Puis, en commentaire : où se concentre le gros des ventes ? La répartition
est-elle symétrique, ou étirée d'un côté ?

> **Rappel.**
> ```python
> df["col"].plot(kind="hist", bins=50, figsize=(7, 4))
> plt.title("...")
> plt.xlabel("...")
> plt.show()
> ```

### Le prix médian par arrondissement

C'est le cœur de la réponse. Il faut une médiane par arrondissement, donc vingt
médianes. Vous savez calculer la médiane d'une colonne ; vous savez garder les
lignes d'un arrondissement avec `query`. Il reste à faire les deux **vingt
fois**, et à ranger les vingt résultats.

#### Une variable dans une condition

`query` travaille sur une chaîne de caractères. Pour y utiliser la valeur d'une
variable Python, on met `@` devant son nom :

In [ ]:
arr = 12
douze = propre.query("arrondissement == @arr")   ## @arr : la valeur de la variable arr

print(len(douze), "ventes dans le 12e, prix median", round(douze["prix_m2"].median()), "euros le m2")

Sans le `@`, pandas chercherait une colonne nommée `arr`. Avec, il lit la
variable. C'est ce qui permet d'écrire la ligne **une fois** et de la faire
tourner pour les vingt arrondissements.

#### La boucle

Le principe est celui de la séance 1.1 : une liste vide avant la boucle, qu'on
remplit avec `append` à chaque tour. Ici il en faut deux, une pour les numéros
d'arrondissement, une pour les médianes.

### Exercice 18 — Vingt médianes

Écrivez une boucle `for` sur `range(1, 21)`. À chaque tour : filtrez `propre` sur
l'arrondissement, calculez la médiane de `prix_m2`, et ajoutez le numéro à la
liste `numeros` et la médiane à la liste `medianes`.

> **Rappel.**
> ```python
> numeros = []
> medianes = []
> for arr in range(1, 21):
>     ...
> ```
> Dans la boucle, le bloc est décalé de quatre espaces.

#### Deux listes, c'est une de trop

Les deux listes vont ensemble : le sixième nombre de `medianes` est la médiane
du sixième numéro de `numeros`. Rien ne le garantit. Triez l'une des deux, et
tout est faux.

pandas a l'objet qu'il faut : une **Series**. C'est une colonne de valeurs où
chaque valeur porte une **étiquette**. Vous en manipulez depuis le début de
l'exercice sans lui avoir donné son nom :

| Ce que vous avez écrit | Les valeurs | Les étiquettes |
|---|---|---|
| `propre["prix_m2"]` | les prix au m² | le numéro de chaque ligne |
| `propre["type_local"].value_counts()` | les effectifs | les noms des types |
| `mois` (exercice 9) | les nombres de ventes | les numéros de mois, d'où `mois.loc[8]` |

On en fabrique une à partir de nos deux listes, les valeurs d'un côté, les
étiquettes de l'autre :

```python
prix_par_arr = pd.Series(medianes, index=numeros)
```

Les deux listes ne font plus qu'un objet, et **l'étiquette voyage avec sa
valeur** : trier, filtrer, tracer, rien ne peut plus les désaligner.

### Exercice 19 — La Series et le graphique

Construisez `prix_par_arr`, affichez-la, puis allez chercher la médiane du 19e
avec `.loc` dans `prix_19`. Enfin tracez-la en barres horizontales, triée, avec
un titre et l'unité.

> **Rappel.** Barres horizontales : `serie.sort_values().plot(kind="barh", figsize=(7, 5))`.
> Le tri croissant met le plus grand en haut. Ajoutez `plt.tight_layout()`
> avant `plt.show()` pour que rien ne soit coupé.

In [ ]:
verifier("vingt arrondissements", len(prix_par_arr) == 20, "la boucle va de 1 a 20 : range(1, 21)")
verifier("mediane du 19e", round(prix_19) == 7836, "pd.Series(medianes, index=numeros), puis .loc[19]")

Les numéros d'arrondissement sont arrivés tout seuls en face des barres :
ce sont les étiquettes de la Series. Avec deux listes, il aurait fallu
expliquer à matplotlib laquelle va où.

Du 19e au 6e, le prix au m² **presque double**. Et le classement n'est pas
une surprise pour un Parisien : l'ouest et le centre en haut, le nord-est en
bas. Vous venez de le mesurer sur 25 000 ventes réelles.

Ce que vous avez écrit dans la boucle, découper selon une colonne, calculer
dans chaque paquet, rassembler les résultats avec leurs étiquettes, pandas
sait le faire en une seule ligne. Vous verrez cette ligne plus tard dans le
cours. Elle vous paraîtra évidente : vous venez d'en écrire le contenu à la
main.

### Exercice 20 — L'année, mois par mois

Même mécanique, sur le temps. Avant d'écrire, une prédiction : **quel mois de
2024 a compté le moins de ventes ?** Vous avez déjà croisé l'information dans
cet exercice.

In [ ]:
# Ma prediction (le mois le plus calme) :

Créez une colonne `mois` dans `propre` à partir de la date. Puis une boucle sur
`range(1, 13)` qui compte les ventes de chaque mois dans une liste. Faites-en
une Series `ventes_par_mois`, étiquetée par le numéro de mois, tracez-la en
courbe, et mettez le mois le plus calme dans `mois_calme`.

> **Rappel.** `df["date"].dt.month` ; `len()` compte les lignes d'un tableau
> filtré ; `serie.plot(kind="line", marker="o", figsize=(7, 4))`.
>
> Pour le mois le plus calme : `serie.sort_values()` trie du plus petit au plus
> grand, et `.index[0]` rend l'étiquette de la première valeur.

In [ ]:
verifier("le mois le plus calme", mois_calme == 8, "sort_values() puis .index[0] : l'etiquette de la plus petite valeur")

Août, sans discussion : deux fois moins de ventes que les mois voisins. Le
marché immobilier part en vacances. Le prix, lui, ne bouge presque pas d'un
mois à l'autre, vous pouvez le vérifier en remplaçant `len(...)` par la
médiane de `prix_m2` dans la boucle. Un marché calme n'est pas un marché qui
baisse.

### Exercice 21 — Surface et prix

Un nuage de points croise deux grandeurs, un point par vente. Tracez le prix
en fonction de la surface. Avant cela, calculez la **corrélation** entre les
deux dans `lien`, arrondie à deux décimales, et mettez-la dans le titre avec un
f-string.

**La corrélation** est un nombre entre -1 et +1. Proche de +1 : quand l'une
monte, l'autre monte. Proche de -1 : quand l'une monte, l'autre descend. Proche
de 0 : pas de lien d'ensemble. Elle met un chiffre sur ce que l'œil croit voir
dans le nuage.

> **Rappel.** `df["a"].corr(df["b"])` ; `df.plot(kind="scatter", x="surface",
> y="prix", alpha=0.2, figsize=(7, 4))`. `alpha` rend les points translucides :
> sans lui, 25 000 points font une tache. Dans un f-string, `{lien}` est
> remplacé par la valeur.

In [ ]:
verifier("correlation", lien == 0.88, "corr() entre surface et prix, arrondi a 2 decimales")

Une corrélation très forte, et le nuage le montre : plus c'est grand, plus
c'est cher. Ce n'est pas une découverte, mais c'est mesuré. Remarquez aussi
l'épaisseur du nuage à surface égale : pour 50 m², les prix vont du simple au
triple. C'est l'arrondissement, et c'est ce que la carte va montrer.

### Exercice 22 — La réponse au client

Le client a 400 000 €. Divisez ce budget par la Series des prix médians :
vous obtenez, pour chaque arrondissement, la surface qu'il peut acheter.
Arrondissez, tracez en barres horizontales triées, et mettez dans `meilleur`
le numéro de l'arrondissement où il achète le plus grand.

> **Rappel.** Une opération sur une Series s'applique à chaque valeur, et les
> étiquettes suivent : `400000 / prix_par_arr` rend une Series. Après
> `sort_values()`, `.index[-1]` est l'étiquette de la plus grande valeur.

In [ ]:
verifier("le meilleur arrondissement", meilleur == 19, "la plus grande surface est la derniere apres un tri croissant : .index[-1]")

Voilà la réponse : **51 m² dans le 19e, 27 m² dans le 6e**, pour le
même budget. Presque du simple au double, à Paris, à quelques stations de
métro d'écart.

---

## Partie 4 — La carte

Le fichier contient la position de chaque appartement vendu : `longitude` et
`latitude`. Un nuage de points avec la longitude en abscisse et la latitude
en ordonnée, c'est **une carte**. Et si on colore chaque point selon son
`prix_m2`, c'est la carte des prix de Paris.

La cellule est écrite pour vous. Le seul mot nouveau est `c=`, qui donne la
colonne qui sert à colorer les points ; `cmap` choisit la palette, et `vmin`,
`vmax` bornent l'échelle des couleurs pour que les extrêmes n'écrasent pas le
reste. Exécutez-la.

In [ ]:
propre.plot(kind="scatter", x="longitude", y="latitude",
            c="prix_m2", cmap="viridis", vmin=6000, vmax=16000,   ## la couleur = le prix au m2
            s=4, alpha=0.6, figsize=(8, 7))
plt.title("Prix au m2 des appartements vendus a Paris en 2024")
plt.xlabel("longitude")
plt.ylabel("latitude")
plt.tight_layout()
plt.show()

C'est Paris. La Seine se devine en creux, le bois de Boulogne à gauche, le
bois de Vincennes à droite. Le centre et l'ouest en jaune, le nord-est en
violet. Chaque point est une vente réelle de 2024, et c'est vous qui les avez
rendues lisibles : au départ, ce fichier ne permettait même pas de calculer un
prix moyen.

Pour finir, la même carte sur un seul arrondissement. Changez le numéro et
exécutez :

In [ ]:
arr_choisi = 11   ## changez le numero

propre.query("arrondissement == @arr_choisi").plot(
    kind="scatter", x="longitude", y="latitude",
    c="prix_m2", cmap="viridis", vmin=6000, vmax=16000,
    s=12, alpha=0.7, figsize=(7, 6))
plt.title(f"Prix au m2 dans le {arr_choisi}e arrondissement (2024)")
plt.tight_layout()
plt.show()

---

## Pour conclure

Complétez cette cellule de texte (double-clic pour l'éditer) en trois phrases,
avec vos chiffres :

- Le prix médian d'un appartement à Paris en 2024 est de … € le m². L'écart entre l'arrondissement le moins cher et le plus cher va de … à … .
- Avec 400 000 €, le client achète le plus grand dans le … arrondissement, soit environ … m².
- Une chose que ce fichier m'a apprise sur le nettoyage des données : …

## Ce que vous avez fait

Vous êtes parti d'un fichier de 61 000 lignes où le prix moyen ne se
calculait même pas. Vous avez :

- compris que chaque ligne était un lot et non une vente, et vérifié à la fin que ce n'était plus le cas ;
- corrigé sept défauts, dans l'ordre, en notant à chaque fois ce que vous retiriez et pourquoi ;
- fait la différence entre réduire son périmètre et perdre des données ;
- reconstruit à la main, avec une boucle et une Series, un calcul par groupe ;
- répondu à la question du client avec un chiffre, un graphique, et une carte.

| Vous avez utilisé | Pour |
|---|---|
| `shape`, `info()`, `head()`, `describe()` | prendre en main un fichier inconnu |
| `len()` contre `nunique()` | savoir ce qu'est une ligne |
| `duplicated()`, `drop_duplicates()` | les doublons |
| `isna().sum()`, `dropna(subset=...)` | les valeurs manquantes, après avoir compris d'où elles venaient |
| `.str.replace()`, `pd.to_numeric(errors="coerce")` | les nombres en texte, et le compte obligatoire après |
| `.str[3:5]`, `pd.to_datetime(format="mixed", dayfirst=True)`, `.dt.month` | les dates, avec un repère avant de convertir |
| `.str.strip().str.lower()` | le texte incohérent |
| `describe()`, `sort_values().head(3)`, `query()` | les valeurs aberrantes |
| `duplicated(subset=..., keep=False)` | les ventes de plusieurs lots |
| une liste, `append`, une boucle `for` | le journal, et les vingt médianes |
| `query("col == @variable")` | une variable dans un filtre |
| `pd.Series(valeurs, index=etiquettes)`, `.loc[etiquette]` | tenir ensemble des valeurs et leurs étiquettes |
| `plot(kind="hist" / "barh" / "line" / "scatter")` | quatre graphiques, quatre questions |
| `corr()` | un chiffre sur un lien |

> ⚠️ **Avant de fermer l'onglet :** vérifiez que votre notebook est bien
> enregistré dans votre Drive.